# 005: Backpropagation Derivation — Chain rule on computation graphs, gradient flow, and numerical verification

## 1. CHAIN RULE ON COMPUTATION GRAPHS
- Backpropagation is the **chain rule of calculus applied systematically** across a computation graph in reverse topological order.
- For a loss $\mathcal{L}$ and a layer $l$:
  $$\frac{\partial \mathcal{L}}{\partial \mathbf{W}^{[l]}} = \frac{\partial \mathcal{L}}{\partial \mathbf{Z}^{[l]}} \cdot \frac{\partial \mathbf{Z}^{[l]}}{\partial \mathbf{W}^{[l]}}$$

## 2. FULL DERIVATION (2-Layer Network)
- **Output layer** ($L = 2$, sigmoid + binary cross-entropy):
  $$d\mathbf{Z}^{[2]} = \mathbf{A}^{[2]} - \mathbf{Y}$$
  $$d\mathbf{W}^{[2]} = \frac{1}{m} d\mathbf{Z}^{[2]} (\mathbf{A}^{[1]})^T$$
  $$d\mathbf{b}^{[2]} = \frac{1}{m} \sum d\mathbf{Z}^{[2]}$$
- **Hidden layer** ($l = 1$, ReLU):
  $$d\mathbf{A}^{[1]} = (\mathbf{W}^{[2]})^T d\mathbf{Z}^{[2]}$$
  $$d\mathbf{Z}^{[1]} = d\mathbf{A}^{[1]} \odot g'^{[1]}(\mathbf{Z}^{[1]})$$
  $$d\mathbf{W}^{[1]} = \frac{1}{m} d\mathbf{Z}^{[1]} (\mathbf{A}^{[0]})^T$$

## 3. GRADIENT CHECKING
- Numerically verify analytical gradients using finite differences:
  $$\frac{\partial \mathcal{L}}{\partial \theta} \approx \frac{\mathcal{L}(\theta + \epsilon) - \mathcal{L}(\theta - \epsilon)}{2\epsilon}$$

---


In [ ]:
import numpy as np

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def relu(z):
    return np.maximum(0, z)

def relu_backward(dA, Z):
    """Derivative of ReLU: pass gradient where Z > 0."""
    return dA * (Z > 0).astype(float)

class BackpropNetwork:
    """2-layer network with full backpropagation from first principles."""
    def __init__(self, n_in, n_hidden, n_out):
        np.random.seed(3)
        self.W1 = np.random.randn(n_hidden, n_in) * np.sqrt(2.0 / n_in)
        self.b1 = np.zeros((n_hidden, 1))
        self.W2 = np.random.randn(n_out, n_hidden) * np.sqrt(2.0 / n_hidden)
        self.b2 = np.zeros((n_out, 1))

    def forward(self, X):
        self.X = X
        self.Z1 = self.W1 @ X + self.b1
        self.A1 = relu(self.Z1)
        self.Z2 = self.W2 @ self.A1 + self.b2
        self.A2 = sigmoid(self.Z2)
        return self.A2

    def compute_loss(self, Y):
        m = Y.shape[1]
        return -(1/m) * np.sum(Y * np.log(self.A2 + 1e-8) + (1-Y) * np.log(1 - self.A2 + 1e-8))

    def backward(self, Y):
        """Full backpropagation derivation."""
        m = Y.shape[1]
        # ─── Output layer gradients ───
        dZ2 = self.A2 - Y                                      # dL/dZ2
        dW2 = (1/m) * dZ2 @ self.A1.T                         # dL/dW2
        db2 = (1/m) * np.sum(dZ2, axis=1, keepdims=True)      # dL/db2
        # ─── Hidden layer gradients (chain rule) ───
        dA1 = self.W2.T @ dZ2                                  # dL/dA1
        dZ1 = relu_backward(dA1, self.Z1)                      # dL/dZ1
        dW1 = (1/m) * dZ1 @ self.X.T                           # dL/dW1
        db1 = (1/m) * np.sum(dZ1, axis=1, keepdims=True)      # dL/db1
        return {'dW1': dW1, 'db1': db1, 'dW2': dW2, 'db2': db2}



In [ ]:
# ── Gradient Checking ──
print("--- Gradient Checking (Numerical vs Analytical) ---")
net = BackpropNetwork(2, 3, 1)
X = np.random.randn(2, 4)
Y = np.array([[1, 0, 1, 0]])

net.forward(X)
grads = net.backward(Y)

epsilon = 1e-7
# Check dW1 element [0, 0]
original_val = net.W1[0, 0]
net.W1[0, 0] = original_val + epsilon
net.forward(X)
loss_plus = net.compute_loss(Y)
net.W1[0, 0] = original_val - epsilon
net.forward(X)
loss_minus = net.compute_loss(Y)
net.W1[0, 0] = original_val

numerical_grad = (loss_plus - loss_minus) / (2 * epsilon)
analytical_grad = grads['dW1'][0, 0]

print(f"Numerical gradient:  {numerical_grad:.8f}")
print(f"Analytical gradient: {analytical_grad:.8f}")
print(f"Relative error:      {abs(numerical_grad - analytical_grad) / (abs(numerical_grad) + abs(analytical_grad) + 1e-8):.2e}")
